In [22]:
import requests
import pandas as pd
import time


In [23]:
API_KEY="8265bd1679663a7ea12ac168da84d2e8"

In [24]:
genre_url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={API_KEY}&language=en-US"

genres = requests.get(genre_url).json()["genres"]

genre_dict = {}

for g in genres:
    genre_dict[g["id"]] = g["name"]

print(genre_dict)

{28: 'Action', 12: 'Adventure', 16: 'Animation', 35: 'Comedy', 80: 'Crime', 99: 'Documentary', 18: 'Drama', 10751: 'Family', 14: 'Fantasy', 36: 'History', 27: 'Horror', 10402: 'Music', 9648: 'Mystery', 10749: 'Romance', 878: 'Science Fiction', 10770: 'TV Movie', 53: 'Thriller', 10752: 'War', 37: 'Western'}


In [ ]:
movies=[]

for page in range(1,501):
    url = f"https://api.themoviedb.org/3/movie/top_rated?api_key={API_KEY}&language=en-US&page={page}"

    data=requests.get(url).json()

    if "results" not in data:
        continue

    for movie in data["results"]:
        genre_names=[genre_dict[g] for g in movie["genre_ids"]]

        movies.append({
            "name":movie["title"],
            "description":movie["overview"],
            "genre":genre_names[0] if genre_names else "Unknown"
        })

print(page)

time.sleep(0.2)
#nut

In [ ]:
df=pd.DataFrame(movies)

In [ ]:
df.to_csv("tmdb_movies_dataset.csv",index=False,encoding="utf-8-sig")

In [ ]:
df.head()

,name,description,genre
0,Accidental Partners,Two women discover they were both scammed by t...,Comedy
1,Swapped,"A small woodland creature and a majestic bird,...",Adventure
2,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama
3,Michael,"The story of Michael Jackson, one of the most ...",Music
4,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   name         10000 non-null  str  
 1   description  10000 non-null  str  
 2   genre        10000 non-null  str  
dtypes: str(3)
memory usage: 3.0 MB


In [ ]:
df.isnull().sum()

name           0
description    0
genre          0
dtype: int64

In [ ]:
df.genre.value_counts()

genre
Drama              2332
Comedy             1936
Action             1282
Horror              833
Animation           592
Adventure           532
Thriller            482
Crime               462
Romance             331
Science Fiction     315
Family              233
Fantasy             225
Mystery             128
War                  98
Western              75
Music                69
History              60
TV Movie             13
Unknown               2
Name: count, dtype: int64

In [ ]:
df=df.dropna()

In [ ]:
df=df.drop_duplicates()

In [ ]:
df.head()

,name,description,genre
0,Accidental Partners,Two women discover they were both scammed by t...,Comedy
1,Swapped,"A small woodland creature and a majestic bird,...",Adventure
2,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,Drama
3,Michael,"The story of Michael Jackson, one of the most ...",Music
4,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",Drama


In [ ]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\XPS\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\XPS\AppData\Roaming\nltk_data...


True

In [ ]:
import re 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words=set(stopwords.words("english"))

lemmatizer=WordNetLemmatizer()

In [ ]:
def clean_text(text):
    text=text.lower()

    text=text.sub(r"http\S+"," ",text)
    text=text.sub(r"[^a-zA-Z]"," ",text)

    words=text.split()

    words=[w for w in words if w not in stop_words]

    words=[lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

In [ ]:
df["clean_description"] = df["description"].apply(clean_text)
df["clean_name"]=df["name"].apply(clean_text)

AttributeError: 'str' object has no attribute 'sub'

In [ ]:
df["text"]=df["clean_name"]+" "+df["clean_description"]

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()

df['label']=encoder.fit_transform(df["genre"])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(max_features=5000)

X=tfdif.fit_transform(df["text"])
Y=df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test=train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model=MultinomialNB()
model.fit(X_train,Y_train)

In [ ]:
from sklearn.metrics import accuracy_score

pred=model.predict(X_test)

print(accuracy_score(Y_test,pred))

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(Y_test,pred))

In [ ]:
from sklearn.metrics import confusion_matrix
cm=confusion_matrix(Y_test,pred)